In [ ]:
library(readr)
library(broom)
library(dplyr)
library(ggplot2)
library(scales)
library(tidyverse)
library(stringr)


In [ ]:
library(partykit)

In [ ]:
install.packages("partykit", repos = "https://cloud.r-project.org", lib = "~/R/library")

In [ ]:
R.version.string


other ideas: 
- https://www.rdocumentation.org/packages/lme4/versions/1.1-38/topics/glmer
- from https://cran.r-project.org/web/packages/lme4/index.html
- check if drug/disease has some effect too

In [ ]:
suffix <- "NEURO" # FULL, NEURO_17, NEURO

file_path <- paste0("out/df_translation_associations_", suffix, ".csv")
ds_name_suffix <- suffix

# LogRegression

In [ ]:
mydata <- read_csv(file_path, show_col_types = FALSE)

mydata$target <- as.factor(mydata$target)
mydata <- mydata[, c(
  "species_unique",
  "strain_unique",
  "assay_unique",
  "tested_both_sexes",
  "articles_total",
  "articles_blinding",
  "articles_randomization",
  "target"
)]
mydata$tested_both_sexes_binary <- factor(
  mydata$tested_both_sexes,
  levels = c(0, 1) # coefficient should represent effect of being tested in both sexes (1 vs 0)
)
mydata$prop_blinding <- mydata$articles_blinding / mydata$articles_total
mydata$prop_randomization <- mydata$articles_randomization / mydata$articles_total


In [ ]:
# Select numeric columns
num_cols <- sapply(mydata, is.numeric)

# Exclude the sex variable
num_cols["tested_both_sexes"] <- FALSE
num_cols["tested_both_sexes_binary"] <- FALSE

# Apply scaling
mydata[, num_cols] <- scale(mydata[, num_cols])

In [ ]:
num_cols

In [ ]:
summary(mydata)


In [ ]:
str(mydata)

## Multivariate

In [ ]:
# =========================
# Toggle: include articles_total?
# =========================
include_articles_total <- FALSE   # set TRUE to include log(articles_total)
exclude_single_article <- FALSE       # set TRUE to drop rows where articles_total == 1

# ---- 1) Optionally filter rows ----
# rows before
rows_before <- nrow(mydata)

# optional filtering
if (exclude_single_article) {
  mydata <- mydata %>%
    filter(articles_total != 1)
}

# rows after
rows_after <- nrow(mydata)

cat("Rows before filtering:", rows_before, "\n")
cat("Rows after filtering :", rows_after, "\n")


# ---- 2) Build formula (optionally include log(articles_total)) ----
base_terms <- c(
  "species_unique",
  "strain_unique",
  "assay_unique",
  "tested_both_sexes_binary",
  "prop_blinding",
  "prop_randomization"
)

if (include_articles_total) {
  base_terms <- c(base_terms, "log(articles_total)")
}

my_formula <- as.formula(paste("target ~", paste(base_terms, collapse = " + ")))

# ---- 3) Fit model ----
mylogit <- glm(
  my_formula,
  data = mydata,
  family = binomial()
)

# ---- 4) Labels (must match tidy() term names) ----
label_map <- c(
  species_unique = "Species (count)",
  strain_unique  = "Strains (count)",
  assay_unique   = "Assays (count)",
  tested_both_sexes_binary1 = "Both sexes (binary)",
  prop_blinding = "Blinding (proportion)",
  prop_randomization = "Randomization (proportion)",
  `log(articles_total)` = "Articles (log)"
)

if (!include_articles_total) {
  label_map <- label_map[names(label_map) != "log(articles_total)"]
}

# ---- 5) Desired ordering: external -> internal -> volume ----
external_terms <- c("species_unique", "strain_unique", "assay_unique", "tested_both_sexes_binary1")
internal_terms <- c("prop_blinding", "prop_randomization")
volume_terms   <- if (include_articles_total) c("log(articles_total)") else character(0)

# ---- 6) Tidy + order for plotting ----
plot_df <- tidy(mylogit, conf.int = TRUE) %>%
  filter(term != "(Intercept)") %>%
  mutate(
    predictor = gsub("`", "", term),

    group = case_when(
      predictor %in% external_terms ~ "External validity",
      predictor %in% internal_terms ~ "Internal validity",
      predictor %in% volume_terms   ~ "Evidence volume",
      TRUE                          ~ "Other"
    ),

    predictor_lab = recode(predictor, !!!label_map),

    or = exp(estimate),
    lo = exp(conf.low),
    hi = exp(conf.high),

    p_lab = ifelse(
      p.value < 0.001,
      "p<0.001",
      paste0("p=", sprintf("%.3f", p.value))
    ),

    within_order = case_when(
      predictor == "species_unique" ~ 1,
      predictor == "strain_unique"  ~ 2,
      predictor == "assay_unique"   ~ 3,
      predictor == "tested_both_sexes_binary1" ~ 4,
      predictor == "prop_blinding"  ~ 5,
      predictor == "prop_randomization" ~ 6,
      predictor == "log(articles_total)" ~ 7,
      TRUE ~ 99
    )
  ) %>%
  arrange(
    factor(group, levels = c("External validity", "Internal validity", "Evidence volume", "Other")),
    within_order
  ) %>%
  mutate(
    predictor_lab = factor(predictor_lab, levels = rev(predictor_lab))
  )

# ---- 7) Plot parameters ----
x_pcol  <- max(plot_df$hi, na.rm = TRUE) * 1.25
x_right <- max(plot_df$hi, na.rm = TRUE) * 1.60

# ---- 8) Forest plot ----
p <- ggplot(plot_df, aes(x = or, y = predictor_lab)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.22, linewidth = 1.0) +
  geom_point(size = 4.0) +
  geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.9, colour = "grey45") +
  geom_text(aes(x = x_pcol, label = p_lab), hjust = 0, size = 7.0) +
  annotate(
    "text",
    x = x_pcol,
    y = length(levels(plot_df$predictor_lab)) + 0.6,
    label = "p-value",
    hjust = 0,
    size = 7.2
  ) +
  scale_x_log10(
    breaks = c(0.25, 0.5, 0.75, 1, 1.5, 2, 4, 8),
    labels = label_number(accuracy = 0.01)
  ) +
  coord_cartesian(
    xlim = c(min(plot_df$lo, na.rm = TRUE) * 0.85, x_right),
    clip = "off"
  ) +
  labs(
    tag = "a",
    title = "Multivariable Logistic Regression",
    x = "Odds ratio (log scale)",
    y = NULL
  ) +
  theme_classic(base_size = 20) +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.3, 0.99),
    plot.title = element_text(
      size = 22,
      face = "plain",
      hjust = 0,
      margin = margin(b = 12)
    ),
    axis.text.y  = element_text(size = 20),
    axis.text.x  = element_text(size = 20),
    axis.title.x = element_text(size = 20, margin = margin(t = 10)),
    plot.margin  = margin(t = 18, r = 20, b = 10, l = 22)
  )

p

# ---- 9) Save ----
suffix <- if (include_articles_total) {
  "with_articles"
} else {
  "no_articles"
}

ggsave(
  filename = paste0("viz/multivariate_effects_", suffix, "_", ds_name_suffix, ".pdf"),
  plot = p,
  width = 9.2,
  height = 5.8,
  units = "in",
  device = cairo_pdf
)

In [ ]:
report_df <- plot_df %>%
  transmute(
    predictor = predictor_lab,
    AOR_CI = sprintf("%.2f (95%% CI %.2f–%.2f)", or, lo, hi),
    p_value = ifelse(p.value < 0.001, "<0.001", sprintf("%.3f", p.value))
  )

report_df

In [ ]:
bad_rows <- plot_df %>%
  mutate(
    bad_or = is.na(or) | is.infinite(or) | or <= 0,
    bad_lo = is.na(lo) | is.infinite(lo) | lo <= 0,
    bad_hi = is.na(hi) | is.infinite(hi) | hi <= 0
  ) %>%
  filter(bad_or | bad_lo | bad_hi)

bad_rows %>% select(term, estimate, conf.low, conf.high, or, lo, hi, p.value)

In [ ]:
exp(coef(mylogit))

In [ ]:
exp(coef(mylogit))

## Univariate

In [ ]:
# ---- 1) Variables (external -> internal -> volume) ----
vars <- c(
  "species_unique",
  "strain_unique",
  "assay_unique",
  "tested_both_sexes_binary",   # stays factor -> term becomes tested_both_sexes_binary1
  "prop_blinding",
  "prop_randomization"
)

if (include_articles_total) {
  vars <- c(vars, "log(articles_total)")
}

# ---- 2) Labels (IMPORTANT: factor term appears as ...1) ----
label_map <- c(
  species_unique = "Species (count)",
  strain_unique  = "Strains (count)",
  assay_unique   = "Assays (count)",
  tested_both_sexes_binary1 = "Both sexes (binary)",
  prop_blinding = "Blinding (proportion)",
  prop_randomization = "Randomization (proportion)",
  `log(articles_total)` = "Articles (log)"
)

if (!include_articles_total) {
  label_map <- label_map[names(label_map) != "log(articles_total)"]
}

# ---- 3) Fit univariate models ----
res <- map_dfr(vars, function(v) {

  m <- glm(
    as.formula(paste("target ~", v)),
    data = mydata,
    family = binomial()
  )

  tidy(m, conf.int = TRUE) %>%
    filter(term != "(Intercept)") %>%
    transmute(
      predictor = gsub("`", "", term),

      estimate,
      conf.low,
      conf.high,
      p.value,

      or = exp(estimate),
      lo = exp(conf.low),
      hi = exp(conf.high),

      p_lab = ifelse(
        p.value < 0.001,
        "p<0.001",
        paste0("p=", sprintf("%.3f", p.value))
      )
    )
})

# ---- 4) Grouping for ordered display ----
external_terms <- c("species_unique", "strain_unique", "assay_unique", "tested_both_sexes_binary1")
internal_terms <- c("prop_blinding", "prop_randomization")
volume_terms   <- if (include_articles_total) c("log(articles_total)") else character(0)

res <- res %>%
  mutate(
    group = case_when(
      predictor %in% external_terms ~ "External validity",
      predictor %in% internal_terms ~ "Internal validity",
      predictor %in% volume_terms   ~ "Evidence volume",
      TRUE                          ~ "Other"
    ),
    predictor_lab = recode(predictor, !!!label_map),

    within_order = case_when(
      predictor == "species_unique" ~ 1,
      predictor == "strain_unique"  ~ 2,
      predictor == "assay_unique"   ~ 3,
      predictor == "tested_both_sexes_binary1" ~ 4,
      predictor == "prop_blinding"  ~ 5,
      predictor == "prop_randomization" ~ 6,
      predictor == "log(articles_total)" ~ 7,
      TRUE ~ 99
    )
  ) %>%
  arrange(
    factor(group, levels = c("External validity", "Internal validity", "Evidence volume", "Other")),
    within_order
  ) %>%
  mutate(
    predictor_lab = factor(predictor_lab, levels = rev(predictor_lab))
  )

# ---- 5) Plot settings ----
x_pcol  <- max(res$hi, na.rm = TRUE) * 1.25
x_right <- max(res$hi, na.rm = TRUE) * 1.60

# ---- 6) Forest plot ----
p <- ggplot(res, aes(x = or, y = predictor_lab)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.22, linewidth = 1.0) +
  geom_point(size = 4.0) +
  geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.9, colour = "grey45") +
  geom_text(aes(x = x_pcol, label = p_lab), hjust = 0, size = 7.0) +
  annotate(
    "text",
    x = x_pcol,
    y = length(levels(res$predictor_lab)) + 0.6,
    label = "p-value",
    hjust = 0,
    size = 7.2
  ) +
  scale_x_log10(
    breaks = c(0.25, 0.5, 0.75, 1, 1.5, 2, 4, 8),
    labels = label_number(accuracy = 0.01)
  ) +
  coord_cartesian(
    xlim = c(min(res$lo, na.rm = TRUE) * 0.85, x_right),
    clip = "off"
  ) +
  labs(
    tag = "b",
    title = "Univariate Logistic Regression",
    x = "Odds ratio (log scale)",
    y = NULL
  ) +
  theme_classic(base_size = 20) +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.3, 0.98),
    plot.title = element_text(size = 22, hjust = 0, margin = margin(b = 12)),
    axis.text.y = element_text(size = 20),
    axis.text.x = element_text(size = 20),
    axis.title.x = element_text(size = 20, margin = margin(t = 10)),
    plot.margin = margin(t = 18, r = 20, b = 10, l = 22)
  )

p

# ---- 7) Save (reflect toggle) ----
suffix <- if (include_articles_total) "with_articles" else "no_articles_neuro"

ggsave(
  filename = paste0("viz/univariate_effects_", suffix, "_", ds_name_suffix, ".pdf"),
  plot = p,
  width = 9,
  height = 5.8,
  units = "in",
  device = cairo_pdf
)

In [ ]:
univ_table <- res %>%
  transmute(
    predictor = as.character(predictor_lab),
    OR_CI = sprintf("%.2f (95%% CI %.2f–%.2f)", or, lo, hi),
    p_value = ifelse(p.value < 0.001, "<0.001", sprintf("%.3f", p.value))
  )

univ_table

# Prediction Experiments

In [ ]:
suffix <- "NEURO_17" # FULL, NEURO_17, NEURO

file_path <- paste0("out/df_translation_associations_", suffix, ".csv")
ds_name_suffix <- suffix

In [ ]:
mydata <- read_csv(file_path, show_col_types = FALSE)

mydata$target <- as.factor(mydata$target)
mydata <- mydata[, c(
  "species_unique",
  "strain_unique",
  "assay_unique",
  "tested_both_sexes",
  "articles_total",
  "articles_blinding",
  "articles_randomization",
  "target"
)]
mydata$tested_both_sexes_binary <- factor(
  mydata$tested_both_sexes,
  levels = c(0, 1) # coefficient should represent effect of being tested in both sexes (1 vs 0)
)
mydata$prop_blinding <- mydata$articles_blinding / mydata$articles_total
mydata$prop_randomization <- mydata$articles_randomization / mydata$articles_total
# rename outcome levels to valid names
# put the positive class first
mydata$target <- factor(mydata$target, levels = c(1, 0), labels = c("yes", "no"))

In [ ]:
dim(mydata)

In [ ]:
mydata_high_quality <- mydata[
  mydata$species_unique > 1 &
  mydata$assay_unique > 1 &
  mydata$tested_both_sexes == 1 &
  mydata$articles_blinding > 0 &
  mydata$articles_randomization > 0,
]

In [ ]:
nrow(mydata_high_quality)

In [ ]:
df <- mydata %>%
  count(target) %>%
  mutate(prop = n / sum(n))

diff_yes_no <- df$prop[df$target == "yes"] - df$prop[df$target == "no"]

diff_yes_no

In [ ]:
df

In [ ]:
df_hq <- mydata_high_quality %>%
  count(target) %>%
  mutate(prop = n / sum(n))

diff_yes_no_hq <- df_hq$prop[df$target == "yes"] - df_hq$prop[df$target == "no"]

diff_yes_no_hq

In [ ]:
(df_hq$prop[df$target == "yes"] - df$prop[df$target == "yes"])*100

In [ ]:
df$prop[df$target == "yes"]

In [ ]:
df_hq

In [ ]:
num_cols <- sapply(mydata, is.numeric)
num_cols["tested_both_sexes"] <- FALSE

mydata[, num_cols] <- lapply(
  mydata[, num_cols],
  function(x) as.numeric(scale(x))
)

In [ ]:
library(caret)

set.seed(123)

train_control <- trainControl(
  method = "cv",
  number = 10,
  classProbs = TRUE,
  summaryFunction = twoClassSummary,
  savePredictions = "final"
)

model_cv <- train(
  target ~ species_unique +
    strain_unique +
    assay_unique +
    tested_both_sexes_binary +
    prop_blinding +
    prop_randomization,
  data = mydata,
  method = "glm",
  family = binomial,
  trControl = train_control,
  metric = "ROC"
)

model_cv

In [ ]:
str(model_cv$pred)

In [ ]:
model_cv$pred
stripchart(yes~obs,data=model_cv$pred)

In [ ]:
library(randomForest)
rf_model <- randomForest(target~species_unique +
                        strain_unique +
                        assay_unique +
                        tested_both_sexes_binary +
                        prop_blinding +
                        prop_randomization,
                        data=mydata)

In [ ]:
prop.table(table(mydata$target))

In [ ]:
rf_model


In [ ]:
cm <- rf_model$confusion

sensitivity <- cm["yes","yes"] / sum(cm["yes", 1:2])
specificity <- cm["no","no"] / sum(cm["no", 1:2])
sensitivity
specificity

In [ ]:
library(comets)


In [ ]:
ggplot(imp, aes(x = imp[[importance_col]], 
                y = reorder(Variable, imp[[importance_col]]))) +
  geom_point(size = 3, color = "#2C7BB6") +
  labs(
    title = "Variable Importance",
    x = "Importance",
    y = NULL
  ) +
  theme_minimal(base_size = 18)

# Stats

## Drug-Disease Pairs

In [ ]:
df_drug_disease <- read_csv(file_path, show_col_types = FALSE)
df_drug_disease$target <- as.factor(df_drug_disease$target)
df_drug_disease <- df_drug_disease[, c(
  "drug_name",
  "articles_total",
  "target"
)]
df_drug_disease <- df_drug_disease %>%
  mutate(
    drug_name = str_replace(drug_name,
                            "^(.*)\\s*<>\\s*(.*)$",
                            "\\2 | \\1")
  )

In [ ]:
nrow(df_drug_disease)

In [ ]:
# ---- 1) Filter and get top 10 ----
top10 <- df_drug_disease %>%
  filter(target == 1) %>%
  arrange(desc(articles_total)) %>%
  slice_head(n = 5) %>%
  mutate(
    drug_name_wrapped = str_wrap(drug_name, width = 40),
    drug_name_wrapped = fct_reorder(drug_name_wrapped, articles_total)
  )

# ---- 2) Plot ----
p <- ggplot(top10, aes(x = articles_total, y = drug_name_wrapped)) +
  geom_col(width = 0.7) +
  labs(
    tag = "c",
    title = "Top 5 Translated Pairs",
    x = "Number of Preclinical Articles",
    y = NULL
  ) +
  theme_classic(base_size = 18) +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.3, 0.98),   # ← tag position you requested
    axis.text.y = element_text(size = 18),
    axis.text.x = element_text(size = 16),
    plot.title  = element_text(size = 20, margin = margin(b = 12)),
    plot.margin = margin(t = 18, r = 20, b = 10, l = 22)
  )

p

# ---- 3) Save ----
ggsave(
  filename = paste0("viz/top5_translated_pairs_articles_", ds_name_suffix, ".pdf"),
  plot = p,
  width = 9,
  height = 5, #7
  units = "in",
  device = cairo_pdf
)

In [ ]:
# ---- 1) Filter and get top 10 ----
top10 <- df_drug_disease %>%
  filter(target == 0) %>%
  arrange(desc(articles_total)) %>%
  slice_head(n = 5) %>%
  mutate(
    drug_name_wrapped = str_wrap(drug_name, width = 40),
    drug_name_wrapped = fct_reorder(drug_name_wrapped, articles_total)
  )

# ---- 2) Plot ----
# ---- 2) Plot ----
p <- ggplot(top10, aes(x = articles_total, y = drug_name_wrapped)) +
  geom_col(width = 0.7) +
  labs(
    tag = "d",
    title = "Top 5 Non-Translated Pairs",
    x = "Number of Preclinical Articles",
    y = NULL
  ) +
  theme_classic(base_size = 18) +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.3, 0.98),
    axis.text.y = element_text(size = 18),
    axis.text.x = element_text(size = 16),
    plot.title = element_text(size = 20, margin = margin(b = 12)),
    plot.margin = margin(t = 18, r = 20, b = 10, l = 22)
  )

p

# ---- 3) Save ----
ggsave(
  filename = paste0("viz/top5_non_translated_pairs_articles_", ds_name_suffix, ".pdf"),
  plot = p,
  width = 9,
  height = 5,
  units = "in",
  device = cairo_pdf
)

## Number of articler per group

In [ ]:
library(dplyr)
df <- read_csv(file_path, show_col_types = FALSE)
df$target <- as.factor(df$target)

df %>%
  summarise(total_all = sum(articles_total, na.rm = TRUE)) 

df %>%
  group_by(target) %>%
  summarise(total_per_group = sum(articles_total, na.rm = TRUE))
summary(df$articles_total)

df <- df %>%
  mutate(
    target = factor(target,
                    levels = c(0, 1),
                    labels = c("Non-translated", "Translated"))
  )

In [ ]:
summary(df$articles_total[df$target == "Translated"])


In [ ]:
summary(df$articles_total[df$target == "Non-translated"])


In [ ]:
sum(is.na(df$articles_total))

In [ ]:
p <- ggplot(df, aes(x = target, y = articles_total)) +
  geom_violin(trim = FALSE, alpha = 0.35) +
  geom_jitter(width = 0.08, alpha = 0.35, size = 1.5) +
  scale_y_log10(
    breaks = c(1, 2, 5, 10, 20, 50, 100),
    labels = label_number()
  ) +
  annotation_logticks(sides = "l") +
  labs(
    tag = "a",
    title = "Distribution of Preclinical Studies",
    x = "",
    y = "Number of Preclinical Articles (log scale)"
  ) +
  theme_classic(base_size = 20) +
  theme(
    plot.tag = element_text(size = 22, face = "bold"),
    plot.tag.position = c(0.05, 0.98),
    plot.title   = element_text(size = 22, hjust = 0.2, margin = margin(b = 12)),
    axis.text.x  = element_text(size = 20),
    axis.text.y  = element_text(size = 20),
    axis.title   = element_text(size = 20),
    plot.margin  = margin(t = 10, r = 20, b = 10, l = 10)
  )

p

ggsave(
  filename = paste0("viz/articles_distribution_by_translation_", ds_name_suffix, ".pdf"),
  plot = p,
  width = 9,
  height = 5.8,
  units = "in",
  device = cairo_pdf
)

In [ ]:
ggsave(
  filename = "viz/articles_total_log2_boxplot.pdf",
  plot = p,
  width = 6,
  height = 5
)


## Species x articles count

In [ ]:
df_filtered <- df %>%
  filter(articles_total == 1, species_unique > 1)



In [ ]:
cor(df$species_unique, df$articles_total, method = "spearman")


In [ ]:
cor.test(df$species_unique, df$articles_total, method = "spearman")


In [ ]:

# Plot
ggplot(df, aes(x = articles_total, y = species_unique)) +
  geom_point(size = 3) +
  labs(
    x = "Articles Count",
    y = "Species Unique",
    title = "Articles Count vs Species Unique"
  ) +
  theme_minimal(base_size = 20)


In [ ]:
df$species_unique_factor <- as.factor(df$species_unique)
label_map <- c(
  species_unique = "Species (count)"
)

mylogit <- glm(
  target ~ species_unique_factor,
  data = df,
  family = binomial()
)

plot_df <- tidy(mylogit, conf.int = TRUE) %>%
  filter(term != "(Intercept)") %>%
  mutate(
    # extract factor level from term like "species_unique_factor3"
    level = str_extract(term, "\\d+"),
    predictor_lab = paste0("Species = ", level),

    or = exp(estimate),
    lo = exp(conf.low),
    hi = exp(conf.high),

    p_lab = ifelse(p.value < 0.001, "p<0.001", paste0("p=", sprintf("%.3f", p.value)))
  ) %>%
  arrange(or) %>%
  mutate(
    predictor_lab = factor(predictor_lab, levels = predictor_lab),
    predictor_lab_wrapped = str_wrap(as.character(predictor_lab), width = 22),
    predictor_lab_wrapped = factor(predictor_lab_wrapped, levels = unique(predictor_lab_wrapped))
  )

# ---- Axis limits based on data only (prevents crammed x-axis) ----
xmin_plot <- min(plot_df$lo, na.rm = TRUE) * 0.85
xmax_plot <- max(plot_df$hi, na.rm = TRUE) * 1.15

# p-value column just outside the plotting region
x_pcol <- xmax_plot * 1.06

p <- ggplot(plot_df, aes(x = or, y = predictor_lab_wrapped)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.18, linewidth = 0.8) +
  geom_point(size = 2.8) +
  geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.7, colour = "grey45") +
  geom_text(aes(x = x_pcol, label = p_lab), hjust = 0, size = 4.5) +
  annotate(
    "text",
    x = x_pcol,
    y = length(levels(plot_df$predictor_lab_wrapped)) + 0.8,
    label = "p-value",
    hjust = 0,
    size = 4.7
  ) +
  scale_x_log10(
    breaks = scales::breaks_log(n = 4),
    labels = label_number(accuracy = 0.01)
  ) +
  coord_cartesian(
    xlim = c(xmin_plot, xmax_plot),
    clip = "off"
  ) +
  labs(
    title = "Logistic Regression (Species)",
    x = "Odds Ratio (log scale)",
    y = NULL
  ) +
  theme_classic(base_size = 16) +
  theme(
    plot.title   = element_text(size = 18, hjust = 0, margin = margin(b = 10)),
    axis.text.y  = element_text(size = 13),
    axis.text.x  = element_text(size = 11),   # smaller to avoid cramming
    axis.title.x = element_text(size = 14, margin = margin(t = 8)),
    plot.margin  = margin(t = 10, r = 130, b = 10, l = 10)  # extra space for p-values
  )

p


ggsave(
  filename = "viz/species_effects.pdf",
  plot = p,
  width = 10,
  height = 5.8,
  units = "in",
  device = cairo_pdf
)
